In [ ]:
import torch
from tqdm.notebook import trange
from llmc.data import CharTokenizer, load_text, train_val_split
from llmc.model import GPT, GPTConfig
from llmc.train import Trainer, TrainConfig

text = load_text("data/tiny_shakespeare.txt")
train_text, val_text = train_val_split(text)
tok = CharTokenizer.from_text(text)
train_ids = torch.tensor(tok.encode(train_text), dtype=torch.long)
val_ids = torch.tensor(tok.encode(val_text), dtype=torch.long)

config = GPTConfig.tiny(vocab_size=tok.vocab_size, block_size=64)
model = GPT(config)
device = "cuda" if torch.cuda.is_available() else "cpu"

trainer = Trainer(
    model,
    train_ids,
    val_ids,
    TrainConfig(max_steps=300, batch_size=32, eval_interval=50, learning_rate=3e-3),
    device=device,
)


In [ ]:
history = trainer.train()
for row in history:
    print(f"step {row['step']:4d} | train {row['train']:.4f} | val {row['val']:.4f}")
